In [ ]:
session.sql("DROP TABLE IF EXISTS tmp_med_hb_stream_snapshot").collect()
print("Temp table dropped")

In [ ]:
session.sql("""
CREATE TEMP TABLE tmp_med_hb_stream_snapshot AS
SELECT *
FROM STREAM_T_MEDICATION_HEALTH_BEHAVIOR
WHERE METADATA$ACTION = 'INSERT'
""").collect()

med_hb_raw = session.table("tmp_med_hb_stream_snapshot")
med_hb_raw = med_hb_raw.drop("METADATA$ACTION", "METADATA$ISUPDATE", "METADATA$ROW_ID")
print(f"snapshot medication_health_behavior records found")

In [ ]:
ind_columns = ["THRIVE_FAILURE_IND", "DATE_UNKNOWN"]

med_hb_clean = med_hb_raw
for c in ind_columns:
    med_hb_clean = med_hb_clean.with_column(
        c,
        when(
            (trim(col(c)).is_null()) | (trim(col(c)) == lit("")),
            lit("N")
        ).otherwise(trim(col(c)))
    )

code_columns = ["PRESCRIBED_FREQUENCY_CODE", "ADMINISTER_METHOD_CODE", "CONSENT_DECISION_CODE"]

for c in code_columns:
    med_hb_clean = med_hb_clean.with_column(
        c,
        upper(trim(col(c)))
    )

desc_columns = ["MEDICATION_CMNT", "DOSAGE_DESC", "CLN_RESP_ADVERSE_EFFECT_CMNT", "CONSENT_COMMENTS"]

for c in desc_columns:
    med_hb_clean = med_hb_clean.with_column(
        c,
        when(
            (trim(col(c)).is_null()) | (trim(col(c)) == lit("")),
            lit("NA")
        ).otherwise(trim(col(c)))
    )

user_columns = ["ADD_USER", "MOD_USER"]

for c in user_columns:
    med_hb_clean = med_hb_clean.with_column(
        c,
        when(
            (trim(col(c)).is_null()) | (trim(col(c)) == lit("")),
            lit("SYSTEM")
        ).otherwise(trim(col(c)))
    )

trim_only_columns = ["SOURCE_FILE_NAME"]

for c in trim_only_columns:
    med_hb_clean = med_hb_clean.with_column(
        c,
        trim(col(c))
    )

med_hb_clean = med_hb_clean.with_column_renamed("LOAD_TIMESTAMP", "RAW_LOAD_TIMESTAMP")
print("med_hb clean")

In [ ]:
valid_med_hb = med_hb_clean.filter(
    col("MHP_ID").is_not_null()
    & col("MED_MED_ID").is_not_null()
    & col("PERSON_PERSON_PATIENT_ID").is_not_null()
)

invalid_med_hb = med_hb_clean.filter(
    col("MHP_ID").is_null()
    | col("MED_MED_ID").is_null()
    | col("PERSON_PERSON_PATIENT_ID").is_null()
).with_column(
    "ERROR_REASON",
    when(col("MHP_ID").is_null(), lit("MHP_ID_NULL"))
    .when(col("MED_MED_ID").is_null(), lit("MED_MED_ID_NULL"))
    .when(col("PERSON_PERSON_PATIENT_ID").is_null(), lit("PERSON_PERSON_PATIENT_ID_NULL"))
    .otherwise(lit("UNKNOWN_ERROR"))
)
print("seperated valid and invalid records")

In [ ]:
checksum_columns = [
    ("MHP_ID", "number"),
    ("MED_MED_ID", "number"),
    ("HPP_HPP_ID", "number"),
    ("PERSON_PERSON_PATIENT_ID", "number"),
    ("PERSON_PERSON_PRESCRIBER_ID", "number"),
    ("PRESCRIBER_PCS_PCS_ID", "number"),
    ("CONSENT_BY_PERSON_PERSON_ID", "number"),
    ("ADD_PERSON_ID", "number"),
    ("ADD_ORGN_ID", "number"),
    ("MOD_PERSON_ID", "number"),
    ("MOD_ORGN_ID", "number"),
    ("THRIVE_FAILURE_IND", "string"),
    ("DATE_UNKNOWN", "string"),
    ("PRESCRIBED_FREQUENCY_CODE", "string"),
    ("ADMINISTER_METHOD_CODE", "string"),
    ("CONSENT_DECISION_CODE", "string"),
    ("MEDICATION_CMNT", "string"),
    ("DOSAGE_DESC", "string"),
    ("CLN_RESP_ADVERSE_EFFECT_CMNT", "string"),
    ("CONSENT_COMMENTS", "string"),
    ("ADD_USER", "string"),
    ("MOD_USER", "string"),
    ("START_DATE", "timestamp"),
    ("END_DATE", "timestamp"),
    ("LAST_REFILL_DATE", "timestamp"),
    ("CONSENT_DATE", "timestamp"),
]

checksum_exprs = []
for col_name, col_type in checksum_columns:
    if col_type == "string":
        checksum_exprs.append(coalesce(col(col_name), lit("")))
    else:
        checksum_exprs.append(coalesce(col(col_name).cast("string"), lit("")))

med_hb_final = valid_med_hb.with_column(
    "CHECKSUM",
    sha2(concat_ws(lit("|"), *checksum_exprs), 256)
)

In [ ]:
med_hb_final = med_hb_final.with_column(
    "STAGING_LOADED_TIMESTAMP",
    current_timestamp()
)

med_hb_final.write.save_as_table(
    table_name=f"{DB}.{STG}.{STG_MEDICATION_HEALTH_BEHAVIOR}",
    mode="truncate"
)

print(f"MED_HB loaded into {STG}.{STG_MEDICATION_HEALTH_BEHAVIOR}")

In [ ]:
invalid_med_hb.create_or_replace_temp_view("tmp_invalid_med_hb")

invalid_count = invalid_med_hb.count()

if invalid_count > 0:
    session.sql(f"""
    INSERT INTO {DB}.{STG}.{INVALID_RECORDS}
    (
        TABLE_NAME,
        ERROR_REASON,
        SOURCE_FILE_NAME,
        RAW_LOAD_TIMESTAMP,
        RAW_RECORD
    )
    SELECT
        'T_MEDICATION_HEALTH_BEHAVIOR',
        ERROR_REASON,
        SOURCE_FILE_NAME,
        RAW_LOAD_TIMESTAMP,
        OBJECT_CONSTRUCT(*)
    FROM tmp_invalid_med_hb
    """).collect()
    print(f"Invalid records saved into {STG}.{INVALID_RECORDS}")
else:
    print("No invalid records")

In [ ]:
rows_processed = med_hb_final.count()
rows_failed = invalid_count

status = 'SUCCESS' if rows_failed == 0 else 'PARTIAL_SUCCESS'

session.call(
    f"{DB}.{AUDIT}.{SP_LOG_AUDIT}",
    session.sql("SELECT UUID_STRING()").collect()[0][0],
    'STG_MEDICATION_HEALTH_BEHAVIOR_LOAD',
    'STAGING',
    datetime.now(),
    datetime.now(),
    rows_processed,
    rows_failed,
    status,
    None
)

session.call(
    f"{DB}.{UTIL}.{SP_SEND_PIPELINE_NOTIFICATION}",
    status,
    'STG_MEDICATION_HEALTH_BEHAVIOR_LOAD',
    'STAGING',
    f'MEDICATION_HEALTH_BEHAVIOR staging completed. Rows processed: {rows_processed}, Failed rows: {rows_failed}'
)
print("Audit log inserted and email sent")